In [5]:
!pip install torchdeq


In [ ]:
import torch
import torch.nn as nn
from torchdeq import get_deq, reset_deq

class SimpleDEQMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        # a small MLP that injects x into the hidden state
        self.fc_x = nn.Linear(input_dim, hidden_dim)
        # a small MLP that updates z
        self.fc_z = nn.Linear(hidden_dim, hidden_dim)
        # instantiate the DEQ solver with default settings
        self.deq = get_deq()  # factory from torchdeq.core :contentReference[oaicite:0]{index=0}
        # final classifier head
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor):
        """
        x: (batch_size, input_dim)
        """
        # compute the "injection" U(x)
        inj = torch.relu(self.fc_x(x))              # → (batch_size, hidden_dim)

        # zero initial state z0
        z0 = torch.zeros_like(inj)                  # → (batch_size, hidden_dim)
                

        # define the DEQ function:  z ↦ φ(W_z z + U(x))
        f = lambda z: torch.relu(self.fc_z(z) + inj)

        # run the fixed-point solver to convergence
        # returns z_star (list of states during solve) and info dict
        z_list, info = self.deq(f, z0)

        # for inference we just take the final estimate
        z_star = z_list[-1]                         # (batch_size, hidden_dim)

        # classification
        return self.classifier(z_star)


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Import torchdeq
from torchdeq import get_deq

# Define the DEQ MLP model
class SimpleDEQMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        # a small MLP that injects x into the hidden state
        self.fc_x = nn.Linear(input_dim, hidden_dim)
        # a small MLP that updates z
        self.fc_z = nn.Linear(hidden_dim, hidden_dim)
        # instantiate the DEQ solver with default settings
        self.deq = get_deq()
        # final classifier head
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor):
        """
        x: (batch_size, input_dim)
        """
        # compute the "injection" U(x)
        inj = torch.relu(self.fc_x(x))              # → (batch_size, hidden_dim)

        # zero initial state z0
        z0 = torch.zeros_like(inj)                  # → (batch_size, hidden_dim)

        # define the DEQ function:  z ↦ φ(W_z z + U(x))
        f = lambda z: torch.relu(self.fc_z(z) + inj)

        # run the fixed-point solver to convergence
        # returns z_star (list of states during solve) and info dict
        z_list, info = self.deq(f, z0)

        # for inference we just take the final estimate
        z_star = z_list[-1]                         # (batch_size, hidden_dim)

        # Debugging print statements
        print(f"Initial residual norm: {torch.norm(z0).item():.4f}")
        print(f"Final residual norm: {torch.norm(z_star).item():.4f}")
        print(f"Solver info: {info}")

        # classification
        return self.classifier(z_star)

# ==== TESTING THE MODEL ====

# Create synthetic data for binary classification
input_dim = 10
hidden_dim = 16
num_classes = 2

X = torch.randn(1000, input_dim)  # 1000 samples, 10 features
y = torch.randint(0, num_classes, (1000,))  # 1000 labels (0 or 1)

# DataLoader
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model, loss, optimizer
model = SimpleDEQMLP(input_dim=input_dim, hidden_dim=hidden_dim, num_classes=num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop
def train_one_epoch():
    model.train()
    for i, (inputs, targets) in enumerate(loader):
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if i % 10 == 0:
            print(f"Batch {i:03d} | Loss: {loss.item():.4f}")

# Run a few epochs
for epoch in range(2):
    print(f"\nEpoch {epoch+1}")
    train_one_epoch()

# Check gradient flow after training
print("\nParameter gradient norms after training:")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"{name}: {param.grad.norm().item():.4f}")



Epoch 1
Initial residual norm: 0.0000
Final residual norm: 10.8149
Solver info: {'abs_lowest': tensor([5.4501e-06, 1.7107e-04, 6.6250e-05, 4.1772e-04, 3.0225e-04, 3.5916e-04,
        4.5258e-04, 2.8082e-04, 5.9063e-04, 8.4507e-05, 3.5252e-04, 8.1342e-04,
        9.2382e-04, 1.4043e-04, 3.3063e-04, 9.7536e-05, 1.3764e-04, 2.3607e-04,
        1.2298e-04, 1.7814e-04, 1.1039e-05, 9.6708e-05, 7.4514e-05, 6.0154e-05,
        3.4497e-04, 9.5085e-05, 6.2939e-07, 7.2545e-04, 1.4914e-05, 2.3025e-05,
        1.8013e-04, 2.7643e-05]), 'rel_lowest': tensor([2.9107e-06, 5.5076e-05, 2.6054e-05, 1.7146e-04, 1.2536e-04, 1.9578e-04,
        6.1556e-04, 1.4380e-04, 5.1787e-04, 3.6355e-05, 3.4701e-04, 4.3190e-04,
        6.0737e-04, 1.0097e-04, 2.0691e-04, 4.4633e-05, 6.4786e-05, 1.1312e-04,
        5.0383e-05, 1.3414e-04, 9.9018e-06, 5.4558e-05, 2.8543e-05, 4.4628e-05,
        1.5786e-04, 6.4093e-05, 4.2436e-07, 4.9543e-04, 7.2729e-06, 2.6001e-05,
        8.0478e-05, 1.3675e-05]), 'abs_trace': tensor([[

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdeq import get_deq, reset_deq

# --- define your CNN update block ---
class CNNBlock(nn.Module):
    def __init__(self, channels: int, groups: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=groups)
        self.norm1 = nn.GroupNorm(num_groups=groups, num_channels=channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=groups)
        self.norm2 = nn.GroupNorm(num_groups=groups, num_channels=channels)

    def forward(self, z: torch.Tensor, inj: torch.Tensor) -> torch.Tensor:
        # one residual‐style CNN step: ReLU( Conv(z) + injection ), then another conv+norm+ReLU
        out = self.conv1(z) + inj
        out = F.relu(self.norm1(out))
        out = self.conv2(out)
        out = F.relu(self.norm2(out))
        return out

# --- the DEQ-CNN model ---
class SimpleDEQCNN(nn.Module):
    def __init__(self, in_channels: int, channels: int, num_classes: int, groups: int = 1):
        super().__init__()
        # U(x): inject input into hidden space
        self.input_conv = nn.Conv2d(in_channels, channels, kernel_size=3, padding=1)
        # implicit CNN block
        self.cnn_block = CNNBlock(channels, groups=groups)
        # DEQ solver
        self.deq = get_deq()
        # final readout
        self.classifier = nn.Linear(channels * 28 * 28, num_classes)

    def forward(self, x: torch.Tensor):
        # x: (B, in_channels, 28, 28)
        # compute injection U(x)
        inj = F.relu(self.input_conv(x))                 # (B, channels, 28,28)
        # initial state z0 = zeros
        z0 = torch.zeros_like(inj)
        # reset any DEQ internals (dropout, norm states, etc.)
        reset_deq(self.deq)

        # define fixed‐point map: z ↦ CNNBlock(z, inj)
        f = lambda z: self.cnn_block(z, inj)

        # run solver
        z_list, info = self.deq(f, z0)
        z_star = z_list[-1]                              # final converged feature map

        # flatten & classify
        out = z_star.view(z_star.size(0), -1)
        return self.classifier(out)


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdeq import get_deq, reset_deq

# --- define your CNN update block ---
class CNNBlock(nn.Module):
    def __init__(self, channels: int, groups: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=groups)
        self.norm1 = nn.GroupNorm(num_groups=groups, num_channels=channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=groups)
        self.norm2 = nn.GroupNorm(num_groups=groups, num_channels=channels)

    def forward(self, z: torch.Tensor, inj: torch.Tensor) -> torch.Tensor:
        out = self.conv1(z) + inj
        out = F.relu(self.norm1(out))
        out = self.conv2(out)
        out = F.relu(self.norm2(out))
        return out

# --- the DEQ‐CNN model ---
class SimpleDEQCNN(nn.Module):
    def __init__(self, in_channels: int, channels: int, num_classes: int, groups: int = 1):
        super().__init__()
        self.input_conv = nn.Conv2d(in_channels, channels, kernel_size=3, padding=1)
        self.cnn_block = CNNBlock(channels, groups=groups)
        self.deq = get_deq()
        self.classifier = nn.Linear(channels * 28 * 28, num_classes)

    def forward(self, x: torch.Tensor):
        inj = F.relu(self.input_conv(x))           # injection U(x)
        z0 = torch.zeros_like(inj)                 # initial state
        reset_deq(self.deq)                        # reset dropout/norm states
        f = lambda z: self.cnn_block(z, inj)       # fixed‐point map
        z_list, info = self.deq(f, z0)             # solve for z*
        z_star = z_list[-1]
        out = z_star.view(z_star.size(0), -1)
        return out, info

if __name__ == "__main__":
    # Instantiate and run on random data
    model = SimpleDEQCNN(in_channels=1, channels=16, num_classes=10, groups=4)
    dummy_input = torch.randn(4, 1, 28, 28)       # batch of 4 random “images”
    logits, solver_info = model(dummy_input)

    print("Logits shape:", logits.shape)         # should be (4, 10)
    print("DEQ solver info:", solver_info)


Logits shape: torch.Size([4, 12544])
DEQ solver info: {'abs_lowest': tensor([6.7782, 5.4006, 6.1266, 7.3451]), 'rel_lowest': tensor([0.0862, 0.0690, 0.0775, 0.0927]), 'abs_trace': tensor([[1.0000e+08, 7.8775e+01, 4.5290e+01, 3.0474e+01, 2.3029e+01, 1.8955e+01,
         1.6477e+01, 1.5070e+01, 1.3996e+01, 1.2890e+01, 1.1784e+01, 1.0957e+01,
         1.0530e+01, 1.0131e+01, 9.8041e+00, 9.4299e+00, 9.1152e+00, 8.9331e+00,
         8.7290e+00, 8.4644e+00, 8.3652e+00, 8.1576e+00, 7.9185e+00, 7.8024e+00,
         7.6211e+00, 7.3490e+00, 7.2695e+00, 7.1402e+00, 7.0773e+00, 7.0548e+00,
         6.9470e+00, 6.8594e+00, 6.8160e+00, 6.7782e+00, 6.8884e+00, 6.9311e+00,
         6.8877e+00, 6.9024e+00, 6.8318e+00, 6.8281e+00, 6.9784e+00],
        [1.0000e+08, 7.8417e+01, 4.5151e+01, 2.9889e+01, 2.2838e+01, 1.8455e+01,
         1.5621e+01, 1.3883e+01, 1.2497e+01, 1.1285e+01, 1.0270e+01, 9.6453e+00,
         9.0928e+00, 8.6290e+00, 8.1293e+00, 7.6402e+00, 7.3897e+00, 7.0374e+00,
         6.7296e+00, 

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdeq import get_deq, reset_deq

class SimpleDEQCNN(nn.Module):
    def __init__(self, in_channels: int, channels: int, num_classes: int, groups: int = 1):
        super().__init__()
        # injection conv
        self.input_conv = nn.Conv2d(in_channels, channels, kernel_size=3, padding=1)
        # implicit DEQ conv layers
        self.deq_conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=groups)
        self.deq_norm1 = nn.GroupNorm(num_groups=groups, num_channels=channels)
        self.deq_conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=groups)
        self.deq_norm2 = nn.GroupNorm(num_groups=groups, num_channels=channels)
        # DEQ solver
        self.deq = get_deq()
        # final classifier
        self.classifier = nn.Linear(channels * 28 * 28, num_classes)

    def forward(self, x: torch.Tensor):
        # 1) compute injection U(x)
        inj = F.relu(self.input_conv(x))           # (B, C, 28, 28)
        # 2) initial z0
        z0 = torch.zeros_like(inj)
        # 3) reset DEQ internal states
        reset_deq(self.deq)
        # 4) define the fixed‐point map f(z) = CNNBlock(z, inj)
        def f(z):
            out = self.deq_conv1(z) + inj
            out = F.relu(self.deq_norm1(out))
            out = self.deq_conv2(out)
            out = F.relu(self.deq_norm2(out))
            return out

        # 5) solve for z* via DEQ
        z_list, info = self.deq(f, z0)
        z_star = z_list[-1]

        # 6) flatten & classify
        out = z_star.view(z_star.size(0), -1)
        logits = self.classifier(out)
        return logits, info

if __name__ == "__main__":
    # quick smoke‐test
    model = SimpleDEQCNN(in_channels=1, channels=16, num_classes=10, groups=4)
    dummy_input = torch.randn(4, 1, 28, 28)
    logits, solver_info = model(dummy_input)
    print("Logits shape:", logits.shape)       # → (4,10)
    print("DEQ info:", solver_info)


Logits shape: torch.Size([4, 10])
DEQ info: {'abs_lowest': tensor([26.5293, 29.5499, 29.5051, 29.7136]), 'rel_lowest': tensor([0.3074, 0.3385, 0.3427, 0.3426]), 'abs_trace': tensor([[1.0000e+08, 8.2458e+01, 4.2586e+01, 3.1983e+01, 2.8342e+01, 2.7375e+01,
         2.6529e+01, 2.7345e+01, 2.9553e+01, 2.9960e+01, 3.0950e+01, 3.1890e+01,
         3.2618e+01, 3.2383e+01, 3.2650e+01, 3.3950e+01, 3.4024e+01, 3.3729e+01,
         3.2777e+01, 3.4555e+01, 3.4800e+01, 3.4151e+01, 3.3853e+01, 3.5058e+01,
         3.5515e+01, 3.4831e+01, 3.5667e+01, 3.5845e+01, 3.5766e+01, 3.5777e+01,
         3.6576e+01, 3.6686e+01, 3.6007e+01, 3.6191e+01, 3.6433e+01, 3.6973e+01,
         3.5603e+01, 3.6580e+01, 3.6513e+01, 3.6490e+01, 3.5199e+01],
        [1.0000e+08, 8.1813e+01, 4.2880e+01, 3.2570e+01, 2.9582e+01, 3.0598e+01,
         2.9858e+01, 2.9550e+01, 2.9754e+01, 3.1541e+01, 3.0497e+01, 3.1989e+01,
         3.2306e+01, 3.4238e+01, 3.3258e+01, 3.3314e+01, 3.3999e+01, 3.5236e+01,
         3.4444e+01, 3.5892